# Seed check: are the intercomparison's differences larger than training noise?

The v3 intercomparison (`intercomparison_90ep_v3`, seed 42) is retrained from scratch, trunk included, with other seeds (`jobs/run_intercomparison.pbs` with `INTERCOMPARISON_SEED`). The replicates train only the emulators the paper's comparisons rest on: deterministic, CRPS ensemble, ConvNeXt U-Net (`samudra`) and ConvNeXt + CRPS (`samudra_crps`).

This notebook reads each run's `intercomparison_summary.csv` and reports:
- every score's mean, standard deviation and range over seeds;
- for each pair compared in the paper, the difference of means and whether the seed ranges overlap (`separated` = every seed of one emulator beats every seed of the other).

With three seeds, `separated` is a coarse test: treat a difference as real only when it is separated, or at least several standard deviations wide.

In [ ]:
# =============================================================================
# 1. Setup: one run per seed
# =============================================================================
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path("/home/156/txs156/uom/OM2-emulator")
sys.path.insert(0, str(REPO_ROOT / "src"))
from Experiment import seed_separation, seed_spread

RUNS = {42: "intercomparison_90ep_v3", 43: "intercomparison_90ep_v3_seed43", 44: "intercomparison_90ep_v3_seed44"}
# Emulators every replicate has (the v3 run also has the supplementary ones).
EMULATORS = ["deterministic", "crps_ensemble", "samudra", "samudra_crps"]
PAIRS = [("crps_ensemble", "samudra_crps"),   # backbone, both CRPS
         ("crps_ensemble", "deterministic"),  # training approach, our UNet
         ("samudra", "deterministic")]        # backbone, both deterministic
FIG_DIR = REPO_ROOT / "figures" / "seed_check_v3"
FIG_DIR.mkdir(parents=True, exist_ok=True)

summaries = {}
for seed, run in RUNS.items():
    path = REPO_ROOT / "figures" / run / "intercomparison_summary.csv"
    if not path.exists():
        print(f"seed {seed}: no {path} yet, skipped")
        continue
    table = pd.read_csv(path, index_col=["variable", "emulator"])
    summaries[seed] = table[table.index.get_level_values("emulator").isin(EMULATORS)]
print(f"seeds: {list(summaries)}")

In [ ]:
# =============================================================================
# 2. Spread over seeds: mean +- std of each score
# =============================================================================
spread = seed_spread(summaries)
spread.to_csv(FIG_DIR / "seed_spread.csv")
scores = spread.columns.get_level_values(0).unique()
compact = pd.DataFrame({score: spread[(score, "mean")].map("{:.3g}".format) + " ± " + spread[(score, "std")].map("{:.2g}".format)
                        for score in scores})
display(compact)

In [ ]:
# =============================================================================
# 3. The paper's comparisons: difference of means, and whether the seed ranges overlap
# =============================================================================
for emulator, other in PAIRS:
    table = seed_separation(summaries, emulator, other)
    table.to_csv(FIG_DIR / f"seed_separation_{emulator}_vs_{other}.csv")
    print(f"\n{emulator} - {other}")
    display(table.round(4))